# E-commerce Business Metrics & Feature Engineering

## Data Science Capstone Project

**Dataset:** E-commerce Behavior Data from Multi-Category Store

**Analysis Period:** October 2019

**Project Date:** August 2026

**Author:** Said Uzun

## Objective

This notebook extends the exploratory analysis by generating business metrics and behavioral features from the e-commerce event data.

The main objectives are:

- Calculate key business metrics from user interactions.
- Create user-, product-, and category-level features.
- Analyze purchasing behavior and engagement patterns.
- Prepare the final feature dataset for recommendation modeling.

## Import Libraries

The required Python libraries are imported for SQL analysis, data manipulation, and visualization.

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

## Data Connection

The raw event dataset is loaded into DuckDB to support efficient analysis of the large event table.

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download \
  -d mkechinov/ecommerce-behavior-data-from-multi-category-store \
  -f 2019-Oct.csv

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
2019-Oct.csv.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -oq 2019-Oct.csv.zip

In [ ]:
!ls -lh

total 9.1G
-rw-r--r-- 1 root root 5.3G Dec  9  2019 2019-Oct.csv
-rw-r--r-- 1 root root 1.7G Dec  9  2019 2019-Oct.csv.zip
drwx------ 5 root root 4.0K Aug 30 10:55 drive
-rw-r--r-- 1 root root 1.4G Aug 30 09:30 ecommerce.db
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 sample_data
-rw-r--r-- 1 root root 767M Aug 30 09:48 user_product_interactions.csv


In [ ]:
db = duckdb.connect("ecommerce.db")

### Loading the Event Data

The October 2019 event dataset is loaded into a DuckDB table for business metric and feature engineering analyses.

In [ ]:
db.sql("""
CREATE OR REPLACE TABLE events AS
SELECT *
FROM read_csv_auto('2019-Oct.csv')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Business Metrics

## Funnel Metrics

This section calculates the main conversion metrics between product views, cart additions, and purchases.

In [ ]:
funnel_metrics = db.sql("""
SELECT
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
FROM events
""").df()

funnel_metrics

,views,carts,purchases
0,40779399.0,926516.0,742849.0


In [ ]:
views = funnel_metrics.loc[0, "views"]
carts = funnel_metrics.loc[0, "carts"]
purchases = funnel_metrics.loc[0, "purchases"]

conversion_metrics = pd.DataFrame({
    "Metric": [
        "View to Cart CR",
        "Cart to Purchase CR",
        "View to Purchase CR"
    ],
    "Rate": [
        carts / views,
        purchases / carts,
        purchases / views
    ]
})

conversion_metrics["Rate"] = conversion_metrics["Rate"].map(lambda x: f"{x:.2%}")

conversion_metrics

,Metric,Rate
0,View to Cart CR,2.27%
1,Cart to Purchase CR,80.18%
2,View to Purchase CR,1.82%


### Key Findings

- The view-to-cart conversion rate is **2.27%**, indicating that only a small share of product views result in cart additions.
- The overall view-to-purchase conversion rate is **1.82%**.
- The cart-to-purchase ratio is **80.18%**, which is unusually high for a strict e-commerce funnel. Since the dataset does not guarantee that every purchase is preceded by a recorded cart event, this metric should be interpreted as an event-level relationship rather than a true sequential conversion rate.

## Purchase Value Metrics

This section analyzes the monetary value associated with purchase events. Since the dataset does not include an explicit order identifier, purchase values are interpreted at event and session levels rather than as traditional order-level revenue metrics.

In [ ]:
purchase_value_metrics = db.sql("""
SELECT
    COUNT(*) AS purchase_events,
    SUM(price) AS total_purchase_value,
    AVG(price) AS avg_purchased_item_price,
    MEDIAN(price) AS median_purchased_item_price
FROM events
WHERE event_type = 'purchase'
""").df()

purchase_value_metrics

,purchase_events,total_purchase_value,avg_purchased_item_price,median_purchased_item_price
0,742849,2.299575e+08,309.561569,179.84


In [ ]:
purchase_session_metrics = db.sql("""
WITH purchase_sessions AS (
    SELECT
        user_session,
        SUM(price) AS session_purchase_value,
        COUNT(*) AS purchased_items
    FROM events
    WHERE event_type = 'purchase'
      AND user_session IS NOT NULL
    GROUP BY user_session
)

SELECT
    COUNT(*) AS purchase_sessions,
    AVG(session_purchase_value) AS avg_purchase_session_value,
    MEDIAN(session_purchase_value) AS median_purchase_session_value,
    AVG(purchased_items) AS avg_items_per_purchase_session
FROM purchase_sessions
""").df()

purchase_session_metrics

,purchase_sessions,avg_purchase_session_value,median_purchase_session_value,avg_items_per_purchase_session
0,629560,365.267015,192.575,1.179949


### Key Findings

- The dataset contains **742,849 purchase events**, representing a total purchase value of approximately **229.96 million**.
- The average purchased item price is **309.56**, while the median purchased item price is **179.84**, indicating a right-skewed price distribution driven by higher-priced products.
- Users complete an average of **1.18 purchased items per purchase session**, suggesting that most purchase sessions involve a single product.
- The average purchase session value is **365.27**, whereas the median is **192.58**, further confirming the presence of a relatively small number of high-value purchase sessions.

# Feature Engineering

This section creates behavioral features to analyze user, product, and category-level interaction patterns.

## User Features

User-level behavioral features are generated to summarize customer activity and purchasing behavior. These features provide a behavioral summary of each user's activity and purchasing patterns.

In [ ]:
user_features = db.sql("""
SELECT
    user_id,

    COUNT(*) AS total_events,

    SUM(CASE WHEN event_type='view' THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type='cart' THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchases,

    COUNT(DISTINCT user_session) AS active_sessions,

    AVG(CASE WHEN event_type='view' THEN price END) AS avg_view_price,
    AVG(CASE WHEN event_type='purchase' THEN price END) AS avg_purchase_price,

    CASE
    WHEN SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) > 0 THEN 1
    ELSE 0
END AS is_buyer

FROM events
GROUP BY user_id
""").df()

user_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,total_events,views,carts,purchases,active_sessions,avg_view_price,avg_purchase_price,is_buyer
0,555487425,42,42.0,0.0,0.0,6,158.727381,NaN,0
1,555488418,1,1.0,0.0,0.0,1,160.620000,NaN,0
2,515104074,6,3.0,2.0,1.0,1,1570.160000,1570.16,1
3,529710990,5,5.0,0.0,0.0,5,61.614000,NaN,0
4,514233738,78,75.0,1.0,2.0,25,450.790133,582.83,1


In [ ]:
user_features.describe()

,user_id,total_events,views,carts,purchases,active_sessions,avg_view_price,avg_purchase_price,is_buyer
count,3.022290e+06,3.022290e+06,3.022290e+06,3.022290e+06,3.022290e+06,3.022290e+06,3.022130e+06,347118.000000,3.022290e+06
mean,5.404674e+08,1.404523e+01,1.349288e+01,3.065609e-01,2.457901e-01,3.058863e+00,3.164080e+02,278.030179,1.148526e-01
std,1.947143e+07,3.277411e+01,3.185573e+01,1.623814e+00,1.409521e+00,6.757155e+00,3.285875e+02,311.215319,3.188441e-01
min,3.386938e+07,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.880000,0.000000e+00
25%,5.205762e+08,2.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.025390e+02,82.427083,0.000000e+00
50%,5.441658e+08,4.000000e+00,4.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,2.068275e+02,172.020000,0.000000e+00
75%,5.582856e+08,1.300000e+01,1.300000e+01,0.000000e+00,0.000000e+00,3.000000e+00,4.012975e+02,331.897143,0.000000e+00
max,5.662809e+08,7.436000e+03,7.436000e+03,4.940000e+02,3.220000e+02,7.400000e+03,2.574070e+03,2574.040000,1.000000e+00


## Product Features

Product-level behavioral features are generated to summarize product popularity, engagement, and purchasing activity.

In [ ]:
product_features = db.sql("""
SELECT
    product_id,

    MEDIAN(price) AS product_price,

    COUNT(*) AS total_events,

    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,

    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT user_session) AS unique_sessions

FROM events
GROUP BY product_id
""").df()

product_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_price,total_events,views,carts,purchases,unique_users,unique_sessions
0,30100077,267.19,55,55.0,0.0,0.0,30,35
1,5700619,48.91,10015,9856.0,2.0,157.0,5927,6656
2,1701218,110.58,346,341.0,1.0,4.0,171,199
3,23000101,8.70,557,556.0,0.0,1.0,404,436
4,1005002,254.54,19076,18001.0,704.0,371.0,9652,11620


In [ ]:
product_features.describe()

,product_id,product_price,total_events,views,carts,purchases,unique_users,unique_sessions
count,1.667940e+05,166794.000000,166794.000000,166794.000000,166794.000000,166794.000000,166794.000000,166794.000000
mean,1.970715e+07,170.470170,254.498147,244.489604,5.554852,4.453691,139.739019,166.588366
std,1.187490e+07,291.030326,3046.366892,2717.840887,226.339217,129.900205,1333.452454,1779.535779
min,1.000978e+06,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
25%,1.090023e+07,22.766250,7.000000,7.000000,0.000000,0.000000,5.000000,5.000000
50%,1.830011e+07,67.100000,25.000000,25.000000,0.000000,0.000000,17.000000,19.000000
75%,2.640381e+07,180.180000,92.000000,91.000000,0.000000,1.000000,58.000000,66.000000
max,6.050001e+07,2574.070000,500354.000000,419287.000000,52123.000000,28944.000000,197878.000000,267574.000000


In [ ]:
product_features["popularity_score"] = (
    product_features["views"] * 1 +
    product_features["carts"] * 3 +
    product_features["purchases"] * 5
)

product_features[
    ["product_id", "views", "carts", "purchases", "popularity_score"]
].sort_values("popularity_score", ascending=False).head(10)

,product_id,views,carts,purchases,popularity_score
83689,1004856,419287.0,52123.0,28944.0,720376.0
83717,1004767,378777.0,37649.0,21806.0,600754.0
83675,1005115,327715.0,15528.0,12543.0,437014.0
83690,1004833,203018.0,21830.0,12697.0,331993.0
83692,4804056,179092.0,22761.0,12381.0,309280.0
83662,1004870,190435.0,19451.0,10615.0,301863.0
83836,1004249,207422.0,14558.0,9090.0,296546.0
73,1002544,179249.0,15799.0,10549.0,279391.0
74,1005105,197930.0,10375.0,7293.0,265520.0
70,5100816,164608.0,17579.0,7273.0,253710.0


## Category Features

Category-level features are generated to summarize engagement and purchasing activity across product categories.

In [ ]:
category_features = db.sql("""
SELECT
    category_code,

    COUNT(*) AS total_events,

    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,

    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT product_id) AS unique_products,

    AVG(price) AS avg_price

FROM events
WHERE category_code IS NOT NULL
GROUP BY category_code
""").df()

category_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category_code,total_events,views,carts,purchases,unique_users,unique_products,avg_price
0,auto.accessories.player,470208,464272.0,1289.0,4647.0,80417,501,137.405998
1,computers.notebook,1137623,1106406.0,15627.0,15590.0,129214,1348,711.357948
2,appliances.kitchen.meat_grinder,160237,155551.0,2304.0,2382.0,27738,258,86.192021
3,appliances.kitchen.hob,106965,104759.0,1247.0,959.0,15217,1171,234.387301
4,electronics.clocks,1311033,1272783.0,20344.0,17906.0,211928,7788,294.445909


In [ ]:
category_features["engagement_score"] = (
    category_features["views"] * 1 +
    category_features["carts"] * 3 +
    category_features["purchases"] * 5
)

category_features[
    ["category_code", "views", "carts", "purchases", "engagement_score"]
].sort_values("engagement_score", ascending=False).head(10)

,category_code,views,carts,purchases,engagement_score
69,electronics.smartphone,10619448.0,549765.0,338018.0,13958833.0
4,electronics.clocks,1272783.0,20344.0,17906.0,1423345.0
77,electronics.audio.headphone,1018542.0,51143.0,30503.0,1324486.0
8,electronics.video.tv,1055961.0,36224.0,21565.0,1272458.0
1,computers.notebook,1106406.0,15627.0,15590.0,1231237.0
70,appliances.kitchen.washer,831279.0,21977.0,16148.0,977950.0
89,appliances.kitchen.refrigerators,863411.0,13126.0,11218.0,958879.0
17,appliances.environment.vacuum,772029.0,17263.0,12378.0,885708.0
9,apparel.shoes,759646.0,0.0,4255.0,780921.0
0,auto.accessories.player,464272.0,1289.0,4647.0,491374.0


## User-Product Interaction Dataset

User interactions are aggregated at the user-product level to create the dataset required for recommendation modeling.

Different interaction types are assigned different weights to reflect their relative importance.

The interaction weights are heuristic values designed to represent increasing levels of user intent: a view receives a weight of 1, a cart addition 3, and a purchase 5.

In [ ]:
user_product_interactions = db.sql("""
SELECT
    user_id,
    product_id,

    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases

FROM events

GROUP BY
    user_id,
    product_id
""").df()

user_product_interactions.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,product_id,views,carts,purchases
0,541312140,44600062,2.0,0.0,0.0
1,554748717,3900821,1.0,0.0,0.0
2,550978835,31500053,1.0,0.0,0.0
3,555158050,2900536,1.0,0.0,0.0
4,530282093,1005011,1.0,0.0,0.0


In [ ]:
user_product_interactions["interaction_score"] = (
    user_product_interactions["views"] * 1
    + user_product_interactions["carts"] * 3
    + user_product_interactions["purchases"] * 5
)

user_product_interactions.head()

,user_id,product_id,views,carts,purchases,interaction_score
0,541312140,44600062,2.0,0.0,0.0,2.0
1,554748717,3900821,1.0,0.0,0.0,1.0
2,550978835,31500053,1.0,0.0,0.0,1.0
3,555158050,2900536,1.0,0.0,0.0,1.0
4,530282093,1005011,1.0,0.0,0.0,1.0


In [ ]:
print("Rows:", len(user_product_interactions))
print("Unique users:", user_product_interactions["user_id"].nunique())
print("Unique products:", user_product_interactions["product_id"].nunique())

Rows: 23307630
Unique users: 3022290
Unique products: 166794


In [ ]:
user_product_interactions.to_csv(
    "user_product_interactions.csv",
    index=False
)

In [ ]:
kpi_summary = db.sql("""
SELECT
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT product_id) AS total_products,
    COUNT(*) AS total_events,
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_events,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events
FROM events
""").df()

kpi_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_users,total_products,total_events,view_events,cart_events,purchase_events
0,3022290,166794,42448764,40779399.0,926516.0,742849.0


In [ ]:
kpi_summary.to_csv(
    "powerbi_kpi_summary.csv",
    index=False
)

print("powerbi_kpi_summary.csv exported successfully.")

powerbi_kpi_summary.csv exported successfully.


In [ ]:
powerbi_funnel = pd.DataFrame({
    "stage": ["View", "Cart", "Purchase"],
    "events": [
        int(funnel_metrics.loc[0, "views"]),
        int(funnel_metrics.loc[0, "carts"]),
        int(funnel_metrics.loc[0, "purchases"])
    ]
})

powerbi_funnel

,stage,events
0,View,40779399
1,Cart,926516
2,Purchase,742849


In [ ]:
powerbi_funnel.to_csv(
    "powerbi_funnel.csv",
    index=False
)

print("powerbi_funnel.csv exported successfully.")

powerbi_funnel.csv exported successfully.


In [ ]:
powerbi_categories = (
    category_features
    .sort_values("engagement_score", ascending=False)
    .head(20)
    .copy()
)

powerbi_categories

,category_code,total_events,views,carts,purchases,unique_users,unique_products,avg_price,engagement_score
69,electronics.smartphone,11507231,10619448.0,549765.0,338018.0,1300236,1285,471.947082,13958833.0
4,electronics.clocks,1311033,1272783.0,20344.0,17906.0,211928,7788,294.445909,1423345.0
77,electronics.audio.headphone,1100188,1018542.0,51143.0,30503.0,213964,2134,97.566032,1324486.0
8,electronics.video.tv,1113750,1055961.0,36224.0,21565.0,170060,557,441.897282,1272458.0
1,computers.notebook,1137623,1106406.0,15627.0,15590.0,129214,1348,711.357948,1231237.0
70,appliances.kitchen.washer,869404,831279.0,21977.0,16148.0,132034,546,331.117489,977950.0
89,appliances.kitchen.refrigerators,887755,863411.0,13126.0,11218.0,131605,1397,397.011284,958879.0
17,appliances.environment.vacuum,801670,772029.0,17263.0,12378.0,119929,753,169.245910,885708.0
9,apparel.shoes,763901,759646.0,0.0,4255.0,126760,6156,89.581890,780921.0
0,auto.accessories.player,470208,464272.0,1289.0,4647.0,80417,501,137.405998,491374.0


In [ ]:
powerbi_categories.to_csv(
    "powerbi_categories.csv",
    index=False
)

print("powerbi_categories.csv exported successfully.")

powerbi_categories.csv exported successfully.


In [ ]:
powerbi_products = (
    product_features
    .sort_values("popularity_score", ascending=False)
    .head(20)
    .copy()
)

powerbi_products

,product_id,product_price,total_events,views,carts,purchases,unique_users,unique_sessions,popularity_score
83689,1004856,131.51,500354,419287.0,52123.0,28944.0,197878,267574,720376.0
83717,1004767,250.66,438232,378777.0,37649.0,21806.0,175611,238638,600754.0
83675,1005115,992.05,355786,327715.0,15528.0,12543.0,171002,234398,437014.0
83690,1004833,172.17,237545,203018.0,21830.0,12697.0,99154,132261,331993.0
83692,4804056,160.55,214234,179092.0,22761.0,12381.0,70528,111018,309280.0
83662,1004870,285.44,220501,190435.0,19451.0,10615.0,84335,115200,301863.0
83836,1004249,740.93,231070,207422.0,14558.0,9090.0,96997,144672,296546.0
73,1002544,460.11,205597,179249.0,15799.0,10549.0,89040,121898,279391.0
74,1005105,1415.48,215598,197930.0,10375.0,7293.0,114823,148080,265520.0
70,5100816,28.63,189460,164608.0,17579.0,7273.0,61368,95396,253710.0


In [ ]:
powerbi_products.to_csv(
    "powerbi_products.csv",
    index=False
)

print("powerbi_products.csv exported successfully.")

powerbi_products.csv exported successfully.


In [ ]:
powerbi_purchase_metrics = pd.DataFrame({
    "purchase_events": [
        int(purchase_value_metrics.loc[0, "purchase_events"])
    ],
    "total_purchase_value": [
        float(purchase_value_metrics.loc[0, "total_purchase_value"])
    ],
    "avg_purchased_item_price": [
        float(purchase_value_metrics.loc[0, "avg_purchased_item_price"])
    ],
    "median_purchased_item_price": [
        float(purchase_value_metrics.loc[0, "median_purchased_item_price"])
    ],
    "purchase_sessions": [
        int(purchase_session_metrics.loc[0, "purchase_sessions"])
    ],
    "avg_purchase_session_value": [
        float(purchase_session_metrics.loc[0, "avg_purchase_session_value"])
    ],
    "median_purchase_session_value": [
        float(purchase_session_metrics.loc[0, "median_purchase_session_value"])
    ],
    "avg_items_per_purchase_session": [
        float(purchase_session_metrics.loc[0, "avg_items_per_purchase_session"])
    ]
})

powerbi_purchase_metrics

,purchase_events,total_purchase_value,avg_purchased_item_price,median_purchased_item_price,purchase_sessions,avg_purchase_session_value,median_purchase_session_value,avg_items_per_purchase_session
0,742849,2.299575e+08,309.561569,179.84,629560,365.267015,192.575,1.179949


In [ ]:
powerbi_purchase_metrics.to_csv(
    "powerbi_purchase_metrics.csv",
    index=False
)

print("powerbi_purchase_metrics.csv exported successfully.")

powerbi_purchase_metrics.csv exported successfully.


# Conclusion

In this notebook, key business metrics were calculated to better understand customer behavior and purchasing patterns. Behavioral features were then engineered at the user, product, category, and user-product levels to transform raw event data into structured analytical datasets.

The resulting interaction dataset captures the strength of user-product relationships through weighted interaction scores and serves as the primary input for the recommendation models developed in the next notebook.

With these features prepared, the project is ready to move from exploratory analysis to recommendation system development.